# Level 2 v3 - Price Forecaster with a proper residual-demand chain
Two models:
1. **Helper** (`model_resid.txt`): forecasts tomorrow's peak residual demand from weather (temp, wind, **solar**) + calendar.
2. **Price** (`model_price.txt`): forecasts price, using the helper's residual-demand estimate.

Why this matters: live, you cannot know tomorrow's actual wind/solar, so we forecast residual demand from tomorrow's weather. We also **validate using predicted residual demand**, so the reported error is honest for live use - no more optimistic backtest.

In [ ]:
# 1. Setup
!pip -q install lightgbm holidays
import pandas as pd, numpy as np, requests, datetime
import matplotlib.pyplot as plt
import lightgbm as lgb, holidays
from sklearn.metrics import mean_absolute_error

## 2. Upload NESO demand CSV(s)

In [ ]:
from google.colab import files
uploaded=files.upload()
raw=pd.concat([pd.read_csv(fn) for fn in uploaded.keys()], ignore_index=True)
print(raw.columns.tolist()); raw.head()

## 3. Actual daily peak residual demand (the helper's training target)

In [ ]:
DATE="SETTLEMENT_DATE"; ND="ND"
WIND="EMBEDDED_WIND_GENERATION"; SOLAR="EMBEDDED_SOLAR_GENERATION"
r=raw[[DATE,ND,WIND,SOLAR]].copy()
r[DATE]=pd.to_datetime(r[DATE], errors="coerce")
for c in [ND,WIND,SOLAR]: r[c]=pd.to_numeric(r[c], errors="coerce")
r=r.dropna(); r["residual"]=r[ND]-r[WIND]-r[SOLAR]
daily=pd.DataFrame({"resid_peak_gw": r.groupby(r[DATE].dt.date)["residual"].max()/1000.0})
daily.index=pd.to_datetime(daily.index); daily=daily.sort_index()
print("range:", daily.index.min().date(),"->",daily.index.max().date()); daily.tail()

## 4. Day-ahead price (Elexon market-index, APX)

In [ ]:
start=daily.index.min().date(); end=daily.index.max().date()
frames=[]; d=start
while d<=end:
    d2=min(d+datetime.timedelta(days=6), end)
    u=("https://data.elexon.co.uk/bmrs/api/v1/balancing/pricing/market-index"
       f"?from={d}T00:00Z&to={d2}T23:59Z&format=json")
    try: frames+=requests.get(u,timeout=60).json().get("data",[])
    except Exception as e: print("fail",d,e)
    d=d2+datetime.timedelta(days=1)
mid=pd.DataFrame(frames); mid=mid[mid["dataProvider"]=="APXMIDP"].copy()
mid["settlementDate"]=pd.to_datetime(mid["settlementDate"]); mid["price"]=pd.to_numeric(mid["price"],errors="coerce")
price_daily=mid.groupby(mid["settlementDate"].dt.date)["price"].max()
price_daily.index=pd.to_datetime(price_daily.index); price_daily=price_daily.rename("price").sort_index()
print("price days:", len(price_daily)); price_daily.tail()

## 5. System price (marginal-cost proxy) - one request per day, takes a couple of minutes

In [ ]:
out={}; d=start
while d<=end:
    u=f"https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/system-prices/{d}?format=json"
    try:
        data=requests.get(u,timeout=60).json().get("data",[])
        vals=[x["systemSellPrice"] for x in data if x.get("systemSellPrice") is not None]
        if vals: out[d]=float(np.mean(vals))
    except Exception: pass
    d=d+datetime.timedelta(days=1)
sys_price=pd.Series(out); sys_price.index=pd.to_datetime(sys_price.index); sys_price=sys_price.rename("sys_price").sort_index()
print("system-price days:", len(sys_price)); sys_price.tail()

## 6. Weather WITH solar radiation and wind (archive)

In [ ]:
lat,lon=51.51,-0.13
ws=daily.index.min().date().isoformat()
we=min(daily.index.max().date(), (pd.Timestamp.today()-pd.Timedelta(days=6)).date()).isoformat()
wurl=("https://archive-api.open-meteo.com/v1/archive"
      f"?latitude={lat}&longitude={lon}&start_date={ws}&end_date={we}"
      "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min,wind_speed_10m_max,shortwave_radiation_sum"
      "&timezone=Europe%2FLondon")
w=requests.get(wurl,timeout=60).json()["daily"]
wx=pd.DataFrame(w); wx["time"]=pd.to_datetime(wx["time"]); wx=wx.set_index("time")
wx.columns=["t_mean","t_max","t_min","wind_max","solar_rad"]; wx.tail()

## 7. Assemble features

In [ ]:
df=daily.join(price_daily,how="inner").join(sys_price,how="inner").join(wx,how="inner")
df["HDD"]=(15.5-df["t_mean"]).clip(lower=0); df["CDD"]=(df["t_mean"]-22).clip(lower=0)
df["dow"]=df.index.dayofweek; df["is_we"]=(df.index.dayofweek>=5).astype(int)
df["month"]=df.index.month; df["doy"]=df.index.dayofyear
uk=holidays.country_holidays("GB", subdiv="ENG")
df["is_hol"]=df.index.to_series().apply(lambda x:int(x in uk))
df["price_lag1"]=df["price"].shift(1); df["sys_lag1"]=df["sys_price"].shift(1)
df=df.dropna()
RESID_FEATURES=["t_mean","t_max","t_min","wind_max","solar_rad","dow","is_we","month","doy","is_hol"]
PRICE_FEATURES=["resid_peak_gw","t_mean","t_max","t_min","wind_max","HDD","CDD","dow","is_we","month","doy","is_hol","price_lag1","sys_lag1"]
print("rows:", len(df)); df.tail()

## 8. Honest validation (helper fit on TRAIN only, price uses PREDICTED residual demand)
This is the key honesty step: the price model is judged using forecast residual demand, exactly as it will run live.

In [ ]:
split=df.index.max()-pd.Timedelta(days=180)
trm=df.index<=split
# helper: weather -> residual demand, fit on train only (no peeking)
helper=lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=31,
                         subsample=0.8, colsample_bytree=0.8, random_state=42)
helper.fit(df.loc[trm, RESID_FEATURES], df.loc[trm,"resid_peak_gw"])
resid_mae=mean_absolute_error(df.loc[~trm,"resid_peak_gw"], helper.predict(df.loc[~trm,RESID_FEATURES]))
print(f"helper residual-demand MAE (test): {resid_mae:.2f} GW")

# price model uses PREDICTED residual demand in place of actual
dfp=df.copy(); dfp["resid_peak_gw"]=helper.predict(df[RESID_FEATURES])
tr,te=dfp[trm], dfp[~trm]
pm=lgb.LGBMRegressor(objective="mae", n_estimators=600, learning_rate=0.03, num_leaves=31,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
pm.fit(tr[PRICE_FEATURES], np.log(tr["price"]))
pred=np.exp(pm.predict(te[PRICE_FEATURES]))
mae=mean_absolute_error(te["price"], pred); base=mean_absolute_error(te["price"], te["price_lag1"])
print(f"Price MAE:    {mae:.2f} GBP/MWh")
print(f"Baseline MAE: {base:.2f} GBP/MWh  -> {100*(1-mae/base):.0f}% vs baseline")

res=te.copy(); res["pred"]=pred; res["ae"]=(res.price-res.pred).abs(); res["be"]=(res.price-res.price_lag1).abs()
for lab,mk in [("normal (<150)",res.price<150),("spike (>=150)",res.price>=150)]:
    s=res[mk]
    print(f"{lab}: {len(s)}d | model {s.ae.mean():.1f} | baseline {s.be.mean():.1f} | {'MODEL WINS' if s.ae.mean()<s.be.mean() else 'baseline wins'}")

## 9. Train BOTH deployment models on all data and save
`model_resid.txt` (weather -> residual demand) and `model_price.txt` (-> price, using predicted residual demand).

In [ ]:
# helper on all data
helper_all=lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=31,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
helper_all.fit(df[RESID_FEATURES], df["resid_peak_gw"])
helper_all.booster_.save_model("model_resid.txt")

# price on all data, using helper's predicted residual demand (train==serve)
dfa=df.copy(); dfa["resid_peak_gw"]=helper_all.predict(df[RESID_FEATURES])
price_all=lgb.LGBMRegressor(objective="mae", n_estimators=600, learning_rate=0.03, num_leaves=31,
                            subsample=0.8, colsample_bytree=0.8, random_state=42)
price_all.fit(dfa[PRICE_FEATURES], np.log(dfa["price"]))
price_all.booster_.save_model("model_price.txt")
print("saved model_resid.txt and model_price.txt")
from google.colab import files
files.download("model_resid.txt"); files.download("model_price.txt")

## Honest notes
- The price validation in cell 8 uses **forecast** residual demand, so its MAE is a realistic estimate of live performance (not optimistic).
- Solar is proxied by one location's radiation - fine for v1, improvable with population weighting.
- Spikes remain event-driven (gas/geopolitics) and outside the features - the pipeline flags those days.